# Prices and markets

How the model makes its **synthetic** wholesale prices, in plain words, one piece at a time.
Every price here is `evidence_kind="synthetic"` and every £ figure is illustrative: none is a
market forecast, a bid, a settlement or Axle cash. Only the demand shapes, the half-hour noise
persistence, the daily-level persistence and the imbalance shares and premiums were calibrated
to a year of Elexon data; section 7 shows that comparison as a reference.

The notebook calls the same functions the app's full run calls
(`sampling.generate_market_price_paths`, `sampling.supply_curve_gbp_per_mwh`,
`action.visible_day_ahead_prices`) and reads every value from `model/assumptions.py`, so a model
change shows up here as a changed chart or a failed check, never as stale text.

**The idea in one paragraph.** Each simulated week has its own *system*: a national demand shape,
extra demand when it is cold, minus wind and solar. What is left is **net demand**, the GW that
gas and other dispatchable plant must cover. A convex **supply curve** turns net demand into a
day-ahead price, so the same extra GW costs more when the system is already tight. A daily level
(gas, carbon, interconnectors: what the net-demand model leaves out) and persistent half-hour
noise sit on top. Random **shocks** add or remove GW at night; some are known the day before and
are in the day-ahead price, the rest are surprises that only move the **intraday** price and the
**imbalance** price. The day-ahead price is set so that, on average, it equals the intraday close
(no free money).

**Sections**

1. Net demand: demand shape, heating, wind and solar
2. The convex supply curve
3. The daily level and the half-hour noise
4. Shocks: known the day before or a surprise
5. Day-ahead, then intraday, then imbalance
6. The no-free-money check
7. Comparison with the Elexon calibration (reference only)
8. Publication at 13:00 the day before, and what a plan may see
9. Sanity checks
10. Limitations

In [1]:
# --- MODEL IMPORTS (one cell) -----------------------------------------------
import re
from datetime import date, datetime, time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from axle_studio.model import assumptions
from axle_studio.model.action import visible_day_ahead_prices
from axle_studio.model.clock import utc_half_hour_boundaries
from axle_studio.model.sampling import (
    SUPPLY_CURVE_INPUTS,
    day_ahead_publication_utc_ns,
    demand_shape_gw,
    generate_market_price_paths,
    sample_daily_temperature,
    solar_daylight_shape,
    supply_curve_gbp_per_mwh,
)
from axle_studio.ui.style import apply_notebook_style

# "notebook_connected" embeds only the figure data and loads plotly.js from a CDN, which keeps the
# committed notebook small. GitHub does not run that script, so every chart has a table under it.
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 100)
LONDON = "Europe/London"

## Try it

Change these and re-run the notebook. `PRICE_OVERRIDES` takes any record name from the table
below, for example `{"wind_installed_gw": 40.0}` (a windier system),
`{"big_shock_rate_per_week": 0.0}` (no big shocks), `{"da_id_premium_gbp_per_mwh": 5.0}`
(day-ahead deliberately £5 below the expected close) or `{"price_cap_gbp_per_mwh": 150.0}`
(a low cap, to see clipping). A July start date switches to the summer shapes and brings more
solar. The generator checks every value, so an impossible one fails loudly.

In [2]:
START_LOCAL_DATE = date(2026, 1, 12)  # the London date the seven study nights start (a Monday)
WORLD_COUNT = 40  # simulated weeks; each has its own weather, wind, solar, shocks and noise
PRICE_OVERRIDES: dict[str, object] = {}

SEED = assumptions.SIMULATION["seed"].value
WARMUP_DAYS = assumptions.SIMULATION["warmup_days"].value
STUDY_DAYS = assumptions.SIMULATION["study_days"].value

In [3]:
# The run inputs the app uses at their defaults: the base temperature per sampled day (warm-up
# first), its SD, and every SYSTEM, SHOCKS, PRICES and TRADING record by name.
inputs = assumptions.forecast_inputs(None, warmup_days=WARMUP_DAYS, study_days=STUDY_DAYS)
price_assumptions = {**inputs["price_assumptions"], **PRICE_OVERRIDES}

price_records = assumptions.assumption_rows().query(
    "section in ['system', 'shocks', 'prices', 'trading']"
)
price_records.loc[:, ["section", "name", "value", "unit", "evidence", "editable"]].reset_index(
    drop=True
)

,section,name,value,unit,evidence,editable
0,system,demand_shape_gw.winter_weekday,"(30.43, -3.07, -1.72, -0.65, -1.43, 0.16, 0.68)","GW Fourier coefficients (a0, a1, b1, a2, b2, a...",illustrative,False
1,system,demand_shape_gw.winter_weekend,"(27.29, -1.35, -2.18, -0.34, -0.43, -0.06, 1.12)","GW Fourier coefficients (a0, a1, b1, a2, b2, a...",illustrative,False
2,system,demand_shape_gw.summer_weekday,"(34.38, -2.0, -1.12, -1.6, -1.85, 0.36, 0.54)","GW Fourier coefficients (a0, a1, b1, a2, b2, a...",illustrative,False
3,system,demand_shape_gw.summer_weekend,"(28.42, 1.45, -3.02, -1.77, -1.61, -0.05, 1.0)","GW Fourier coefficients (a0, a1, b1, a2, b2, a...",illustrative,False
4,system,summer_months,"(4, 5, 6, 7, 8, 9)",month numbers,illustrative,False
5,system,heating_threshold_c,15.5,degrees C,illustrative,False
6,system,heating_gw_per_c,0.6,GW per degree C below the threshold,illustrative,True
7,system,wind_installed_gw,30.0,GW,illustrative,True
8,system,wind_median_capacity_factor.winter,0.4,fraction,illustrative,False
9,system,wind_median_capacity_factor.summer,0.28,fraction,illustrative,False


In [4]:
# Prices cover the warm-up and the study (decision 0004 item 52): the warm-up gives the charger
# a week of published history and the trading baseline its reference nights.
run_starts = pd.DatetimeIndex(
    utc_half_hour_boundaries(START_LOCAL_DATE, WARMUP_DAYS, STUDY_DAYS)[:-1]
)
SLOT_COUNT = len(run_starts)
STUDY = slice(48 * WARMUP_DAYS, None)
london = run_starts.tz_convert(LONDON)
london_date = np.asarray(london.date)
# Day index of each slot's London date, counted from the first priced date (column 0 of the
# temperature array, as the generator reads it).
day_index = np.asarray([(day - london_date[0]).days for day in london_date])


def generate_prices(overrides=None):
    """Temperatures (world, day) in deg C and the `MarketPrices` of one seeded set of weeks.

    A fresh `default_rng(SEED)` each call, so two calls with different overrides see the same
    random futures (common random numbers): any difference between them is the edited term alone.
    A full app run draws the population and trips from its generator first, so its prices differ
    from these, but the mechanism is the same.
    """
    rng = np.random.default_rng(SEED)
    temperature_c = sample_daily_temperature(
        rng,
        world_count=WORLD_COUNT,
        base_temperature_c_by_day=inputs["base_temperature_c_by_day"],
        sd_c=inputs["weather_sd_c"],
    )
    market = generate_market_price_paths(
        rng,
        interval_start_utc=run_starts,
        evaluation_world_count=WORLD_COUNT,
        daily_temperature_c=temperature_c,
        price_assumptions={**price_assumptions, **(overrides or {})},
    )
    return temperature_c, market


def as_matrix(frame: pd.DataFrame, column: str) -> np.ndarray:
    """A world-major long price frame column as a (world, slot) array."""
    return frame[column].to_numpy(dtype=float).reshape(WORLD_COUNT, SLOT_COUNT)


temperature_c, market = generate_prices()
day_ahead = as_matrix(market.day_ahead, "wholesale_forecast_gbp_per_mwh")
net_demand = as_matrix(market.day_ahead, "system_net_demand_gw")
close = as_matrix(market.realised, "evaluation_context_price_gbp_per_mwh")
imbalance = as_matrix(market.realised, "imbalance_price_gbp_per_mwh")
system_long = market.realised["system_long"].to_numpy().reshape(WORLD_COUNT, SLOT_COUNT)
pd.DataFrame(
    {
        "worlds": [WORLD_COUNT],
        "priced slots per world": [SLOT_COUNT],
        "study slots per world": [SLOT_COUNT - 48 * WARMUP_DAYS],
        "shocks drawn": [len(market.shocks)],
        "evidence_kind": [market.day_ahead["evidence_kind"].iat[0]],
    }
)

,worlds,priced slots per world,study slots per world,shocks drawn,evidence_kind
0,40,672,336,1562,synthetic


## 1. Net demand: demand shape, heating, wind and solar

**Demand shape.** National demand before wind and solar, as a smooth daily curve with three
harmonics (`sampling.demand_shape_gw`). There are four shapes (winter/summer x weekday/weekend);
the run's first London month picks the season for the whole week, so a week that crosses 1 April
does not jump. The coefficients were fitted backwards from the Elexon price shapes: after heating,
wind and solar are applied and the result goes through the supply curve, the simulated mean price
of each half-hour matches the observed one (the `_DEMAND_SHAPES` comment in `assumptions.py`
explains how). So they are shapes that reproduce the observed price pattern, not metered demand.

**Heating.** Each deg C a day is below the heating threshold adds a fixed number of GW for the whole
day. Most GB homes heat with gas, so the electric effect is a few GW. The temperature is the run's
own sampled daily temperature, so a cold simulated week is also a dear one.

**Wind.** Installed GW x a daily capacity factor. The factor lives on the logit scale with an AR(1)
across days, so it stays between 0 and 1 and calm or windy spells last several days.

**Solar.** Installed GW x a clear-sky daylight shape (a half sine between sunrise and sunset) x a
daily clearness, also a persistent AR(1) on the logit scale. Zero at night.

Net demand = demand + heating - wind - solar (+ any shock known the day before, section 4).

In [5]:
hours = np.arange(48) / 2
season_names = ("winter_weekday", "winter_weekend", "summer_weekday", "summer_weekend")
shape_figure = go.Figure()
for name in season_names:
    shape_figure.add_trace(
        go.Scatter(
            x=hours,
            y=demand_shape_gw(price_assumptions[f"demand_shape_gw.{name}"], hours),
            mode="lines",
            name=name.replace("_", " "),
        )
    )
for season in ("winter", "summer"):
    shape_figure.add_trace(
        go.Scatter(
            x=hours,
            y=price_assumptions["solar_installed_gw"]
            * solar_daylight_shape(hours, price_assumptions, season),
            mode="lines",
            line={"dash": "dot"},
            name=f"clear-sky solar, {season}",
        )
    )
shape_figure.update_layout(
    title="Demand shapes and clear-sky solar by London clock hour (illustrative records)",
    xaxis_title="London clock hour",
    yaxis_title="GW",
)
apply_notebook_style(shape_figure, height=420)

In [6]:
def clock(hour: float) -> str:
    """A clock hour such as 17.5 as '17:30'."""
    return f"{int(hour):02d}:{round(hour % 1 * 60):02d}"


shape_table = pd.DataFrame(
    [
        {
            "shape": name,
            "min GW": (
                values := demand_shape_gw(price_assumptions[f"demand_shape_gw.{name}"], hours)
            ).min(),
            "trough at": clock(hours[values.argmin()]),
            "max GW": values.max(),
            "peak at": clock(hours[values.argmax()]),
        }
        for name in season_names
    ]
).round(2)
shape_table

,shape,min GW,trough at,max GW,peak at
0,winter_weekday,25.96,02:30,33.82,10:00
1,winter_weekend,24.16,05:30,30.93,18:00
2,summer_weekday,30.23,02:00,37.64,18:00
3,summer_weekend,24.79,13:00,34.79,19:00


**Taking net demand apart without copying the model.** The generator returns only the total net
demand. To see each term, the notebook calls it again with the same seed and one term switched on
at a time (heating, wind or solar installed at zero; no shocks). Because every run draws the same
random numbers in the same order, the difference between two runs is exactly the term that was
switched. The check below confirms the terms add back up to the full run's net demand.

In [7]:
terms_off = {
    "heating_gw_per_c": 0.0,
    "wind_installed_gw": 0.0,
    "solar_installed_gw": 0.0,
    "mild_shock_rate_per_night": 0.0,
    "big_shock_rate_per_week": 0.0,
}


def net_demand_with(**switched_on):
    """Net demand (world, slot) GW: every term off except those named, which keep their records."""
    overrides = {name: value for name, value in terms_off.items() if name not in switched_on}
    return as_matrix(generate_prices(overrides)[1].day_ahead, "system_net_demand_gw")


demand_gw = net_demand_with()
heating_gw = net_demand_with(heating_gw_per_c=True) - demand_gw
wind_gw = demand_gw - net_demand_with(wind_installed_gw=True)
solar_gw = demand_gw - net_demand_with(solar_installed_gw=True)
decomposition_gap_gw = np.abs(
    demand_gw + heating_gw - wind_gw - solar_gw + market.known_shock_gw - net_demand
).max()
print(f"largest gap between the terms and the full net demand: {decomposition_gap_gw:.2e} GW")

largest gap between the terms and the full net demand: 1.42e-14 GW


In [8]:
world = 0
study_local = london[STUDY]
components = {
    "demand shape": demand_gw,
    "heating": heating_gw,
    "wind (subtracted)": wind_gw,
    "solar (subtracted)": solar_gw,
    "net demand": net_demand,
}
component_figure = go.Figure()
for name, values in components.items():
    component_figure.add_trace(
        go.Scatter(
            x=study_local,
            y=values[world, STUDY].round(3),
            mode="lines",
            name=name,
            line={"width": 3 if name == "net demand" else 1.5},
        )
    )
component_figure.update_layout(
    title=f"Net demand and its terms, simulated week {world} (synthetic)",
    xaxis_title="London local time",
    yaxis_title="GW",
)
apply_notebook_style(component_figure, height=440)

In [9]:
study_days_index = day_index[STUDY]
daily_components = (
    pd.DataFrame(
        {
            "London date": london_date[STUDY],
            **{name: values[world, STUDY] for name, values in components.items()},
        }
    )
    .groupby("London date")
    .mean()
)
daily_components.insert(
    0, "temperature °C", temperature_c[world, np.unique(study_days_index)][: len(daily_components)]
)
daily_components.round(2)

,temperature °C,demand shape,heating,wind (subtracted),solar (subtracted),net demand
London date,,,,,,
2026-01-12,7.37,31.50,4.88,6.44,0.68,29.01
2026-01-13,8.97,30.43,3.92,7.28,0.94,26.06
2026-01-14,9.29,30.43,3.72,9.99,1.24,22.93
2026-01-15,13.76,30.43,1.04,13.63,1.15,16.69
2026-01-16,11.56,30.43,2.37,9.09,1.17,23.29
2026-01-17,7.13,27.29,5.02,7.38,1.06,24.31
2026-01-18,11.25,27.29,2.55,10.85,0.50,18.53
2026-01-19,9.94,29.36,3.34,13.24,0.17,19.28


**Why a cold world is a dear world.** Across every simulated day, heating is zero above the
threshold and rises by the same GW per deg C below it: a hinge. The table bins the same points.

In [10]:
first_slot_of_day = np.unique(day_index, return_index=True)[1]
hinge = pd.DataFrame(
    {
        "temperature_c": temperature_c[:, day_index[first_slot_of_day]].ravel(),
        "heating_gw": heating_gw[:, first_slot_of_day].ravel(),
    }
)
hinge_figure = go.Figure(
    go.Scatter(
        x=hinge["temperature_c"].round(2),
        y=hinge["heating_gw"].round(3),
        mode="markers",
        marker={"size": 5, "opacity": 0.6},
        name="one simulated day",
    )
)
hinge_figure.add_vline(
    x=price_assumptions["heating_threshold_c"],
    line_dash="dash",
    annotation_text="heating threshold (record)",
)
hinge_figure.update_layout(
    title="Daily temperature against the heating term, every simulated day",
    xaxis_title="Daily temperature (°C)",
    yaxis_title="Heating (GW)",
)
apply_notebook_style(hinge_figure, height=380)

In [11]:
hinge.assign(temperature_band=pd.cut(hinge["temperature_c"], bins=8)).groupby(
    "temperature_band", observed=True
)["heating_gw"].agg(["count", "mean", "min", "max"]).round(2)

,count,mean,min,max
temperature_band,,,,
"(3.321, 4.847]",22,6.76,6.40,7.30
"(4.847, 6.361]",76,5.84,5.49,6.39
"(6.361, 7.876]",137,4.98,4.58,5.48
"(7.876, 9.39]",143,4.13,3.67,4.57
"(9.39, 10.904]",117,3.24,2.77,3.66
"(10.904, 12.419]",74,2.36,1.86,2.75
"(12.419, 13.933]",20,1.47,1.00,1.82
"(13.933, 15.447]",11,0.60,0.03,0.88


## 2. The convex supply curve

`supply_curve_gbp_per_mwh` turns net demand into a price:
a straight line (the gas-set middle of the merit order; below zero at very low net demand, where
inflexible and subsidised plant pays to keep running) plus an exponential scarcity term that is
almost nothing below the scarcity threshold and steep above it. Because the curve is convex,
**the same extra GW moves the price more when the system is already tight** (decision 0004
items 55 and 56). That is why a cold, still evening is where shocks hurt.

The dots are simulated half-hours. They scatter around the curve because the day-ahead price also
carries the daily level, the half-hour noise and the expected cost of surprises (sections 3-6).

In [12]:
curve_values = {name: price_assumptions[name] for name in SUPPLY_CURVE_INPUTS}
grid_gw = np.linspace(np.floor(net_demand.min()) - 2, np.ceil(net_demand.max()) + 3, 200)
curve_figure = go.Figure()
sample_worlds = slice(0, 5)
curve_figure.add_trace(
    go.Scatter(
        x=net_demand[sample_worlds, STUDY].ravel().round(3),
        y=day_ahead[sample_worlds, STUDY].ravel().round(2),
        mode="markers",
        marker={"size": 3, "opacity": 0.35},
        name="simulated day-ahead half-hours (5 weeks)",
    )
)
curve_figure.add_trace(
    go.Scatter(
        x=grid_gw,
        y=supply_curve_gbp_per_mwh(grid_gw, **curve_values),
        mode="lines",
        line={"width": 3},
        name="supply curve (records)",
    )
)
curve_figure.add_vline(
    x=curve_values["scarcity_threshold_gw"], line_dash="dash", annotation_text="scarcity threshold"
)
curve_figure.update_layout(
    title="Supply curve and simulated day-ahead prices",
    xaxis_title="Net demand (GW)",
    yaxis_title="£/MWh",
)
apply_notebook_style(curve_figure, height=440)

**Try it:** change `SHOCK_GW`. The table adds the same shock at a quiet, a typical, a busy and a
tight net demand (quantiles of the simulated study net demand, plus the scarcity threshold).

In [13]:
SHOCK_GW = 2.0
study_net = net_demand[:, STUDY].ravel()
base_levels = {
    "P10 net demand": np.quantile(study_net, 0.10),
    "P50 net demand": np.quantile(study_net, 0.50),
    "P90 net demand": np.quantile(study_net, 0.90),
    "P99 net demand": np.quantile(study_net, 0.99),
    "scarcity threshold": curve_values["scarcity_threshold_gw"],
}
increments = pd.DataFrame(
    [
        {
            "starting point": name,
            "net demand GW": level,
            "price £/MWh": supply_curve_gbp_per_mwh(level, **curve_values),
            f"price after +{SHOCK_GW:g} GW": supply_curve_gbp_per_mwh(
                level + SHOCK_GW, **curve_values
            ),
        }
        for name, level in base_levels.items()
    ]
).sort_values("net demand GW")
increments["price change £/MWh"] = (
    increments[f"price after +{SHOCK_GW:g} GW"] - increments["price £/MWh"]
)
increments.round(2).reset_index(drop=True)

,starting point,net demand GW,price £/MWh,price after +2 GW,price change £/MWh
0,P10 net demand,13.31,44.59,54.62,10.03
1,P50 net demand,21.44,85.75,96.25,10.50
2,P90 net demand,28.38,125.27,140.35,15.08
3,P99 net demand,32.52,161.86,192.03,30.17
4,scarcity threshold,33.00,168.00,201.69,33.69


## 3. The daily level and the half-hour noise

Two things the net-demand model leaves out are added to the curve price:

- a **daily level** that moves every half-hour of a London day together (gas and carbon prices,
  interconnector flows, outages), an AR(1) across days, so dear days follow dear days;
- **half-hour noise**, an AR(1) along the half-hours, because real prices persist: independent
  noise would make a cheap slot beside an expensive one far too common, and a smart charger would
  find bargains that do not exist.

Same trick as section 1: two more runs with the same seed, one with the noise SD at 0 and one
with the level SD at 0. The differences from the full run are the noise and the level.

In [14]:
noise_gbp = day_ahead - as_matrix(
    generate_prices({"day_ahead_sd_gbp_per_mwh": 0.0})[1].day_ahead,
    "wholesale_forecast_gbp_per_mwh",
)
level_gbp = day_ahead - as_matrix(
    generate_prices({"daily_level_sd_gbp_per_mwh": 0.0})[1].day_ahead,
    "wholesale_forecast_gbp_per_mwh",
)
level_spread_within_day = max(
    np.ptp(level_gbp[:, day_index == day], axis=1).max() for day in np.unique(day_index)
)
print(f"largest spread of the level inside one London day: £{level_spread_within_day:.4f}/MWh")

noise_figure = go.Figure()
noise_figure.add_trace(
    go.Scatter(x=study_local, y=level_gbp[world, STUDY].round(2), mode="lines", name="daily level")
)
noise_figure.add_trace(
    go.Scatter(
        x=study_local, y=noise_gbp[world, STUDY].round(2), mode="lines", name="half-hour noise"
    )
)
noise_figure.update_layout(
    title=f"Daily level and half-hour noise added to the curve price, simulated week {world}",
    xaxis_title="London local time",
    yaxis_title="£/MWh",
)
apply_notebook_style(noise_figure, height=400)

largest spread of the level inside one London day: £0.0046/MWh


In [15]:
def lag_one(series: np.ndarray) -> float:
    """Mean over worlds of each world's lag-one autocorrelation along the last axis."""
    return float(np.mean([np.corrcoef(row[:-1], row[1:])[0, 1] for row in series]))


daily_level = level_gbp[:, first_slot_of_day]
noise_level_table = pd.DataFrame(
    [
        {
            "component": "half-hour noise",
            "measured SD £/MWh": noise_gbp.std(),
            "record SD £/MWh": price_assumptions["day_ahead_sd_gbp_per_mwh"],
            "measured lag-one": lag_one(noise_gbp),
            "record AR(1)": assumptions.PRICES["ar1_phi"].value,
            "points per world": SLOT_COUNT,
        },
        {
            "component": "daily level",
            "measured SD £/MWh": daily_level.std(),
            "record SD £/MWh": price_assumptions["daily_level_sd_gbp_per_mwh"],
            "measured lag-one": lag_one(daily_level),
            "record AR(1)": price_assumptions["daily_level_ar1_phi"],
            "points per world": daily_level.shape[1],
        },
    ]
).round(3)
noise_level_table

,component,measured SD £/MWh,record SD £/MWh,measured lag-one,record AR(1),points per world
0,half-hour noise,24.148,24.0,0.914,0.92,672
1,daily level,4.822,5.0,0.437,0.64,15


The daily level's lag-one estimate sits below its record because each world has only about 15
days, and a short series underestimates persistence; the half-hour noise, with hundreds of points
per world, matches its record closely.

## 4. Shocks: known the day before or a surprise

Besides the smooth terms, net demand carries random **shocks** (decision 0004 item 56): frequent
mild ones and a few big ones a week, up (a cold still evening, a plant trip) or down (a windy
mild night). They start only inside the nightly window (London 15:00 to 09:00), the hours that
matter for home charging, ramp up and down as a trapezoid, and pass through the convex curve, so
the same GW moves price more when the system is tight.

Each shock is either **known day-ahead** (it is in the net demand the day-ahead auction clears
on) or a **surprise** (revealed a couple of hours before it starts, so it moves only the intraday
price, and imbalance if revealed after gate closure). The candidates are drawn in a fixed number
per night and thinned by the rate, so editing a rate never moves another random draw.

In [16]:
shocks = market.shocks
night_count = WARMUP_DAYS + STUDY_DAYS  # one 15:00-09:00 window per noon-to-noon day
shock_records = assumptions.SHOCKS
shock_table = pd.DataFrame(
    [
        {
            "class": "mild",
            "simulated rate": (shocks["shock_class"] == "mild").sum() / (WORLD_COUNT * night_count),
            "record rate": price_assumptions["mild_shock_rate_per_night"],
            "rate unit": shock_records["mild_shock_rate_per_night"].unit,
            "simulated known share": shocks.query("shock_class == 'mild'")[
                "known_day_ahead"
            ].mean(),
            "record known share": price_assumptions["mild_shock_known_share"],
            "simulated mean size GW": shocks.query("shock_class == 'mild'")["size_gw"].mean(),
            "simulated median size GW": shocks.query("shock_class == 'mild'")["size_gw"].median(),
            "record size GW": price_assumptions["mild_shock_scale_gw"],
            "record size meaning": "exponential scale (its mean)",
        },
        {
            "class": "big",
            "simulated rate": (shocks["shock_class"] == "big").sum()
            / (WORLD_COUNT * night_count / 7),
            "record rate": price_assumptions["big_shock_rate_per_week"],
            "rate unit": shock_records["big_shock_rate_per_week"].unit,
            "simulated known share": shocks.query("shock_class == 'big'")["known_day_ahead"].mean(),
            "record known share": price_assumptions["big_shock_known_share"],
            "simulated mean size GW": shocks.query("shock_class == 'big'")["size_gw"].mean(),
            "simulated median size GW": shocks.query("shock_class == 'big'")["size_gw"].median(),
            "record size GW": price_assumptions["big_shock_median_gw"],
            "record size meaning": "lognormal median",
        },
    ]
)
shock_table.round(3)

,class,simulated rate,record rate,rate unit,simulated known share,record known share,simulated mean size GW,simulated median size GW,record size GW,record size meaning
0,mild,2.052,2.0,shocks per night,0.484,0.5,1.028,0.710,1.0,exponential scale (its mean)
1,big,5.162,5.0,shocks per week,0.387,0.4,4.647,4.048,4.0,lognormal median


In [17]:
shock_start_hour = shocks["start_utc"].dt.tz_convert(LONDON)
shocks_by_hour = (
    shocks.assign(start_hour=shock_start_hour.dt.hour)
    .pivot_table(index="start_hour", columns="shock_class", values="size_gw", aggfunc="size")
    .reindex(range(24), fill_value=0)
    .fillna(0)
    .astype(int)
)
hour_figure = go.Figure()
for shock_class in shocks_by_hour.columns:
    hour_figure.add_trace(
        go.Bar(x=shocks_by_hour.index, y=shocks_by_hour[shock_class], name=f"{shock_class} shocks")
    )
hour_figure.update_layout(
    barmode="stack",
    title="Shock start times by London hour: only inside the 15:00-09:00 window",
    xaxis_title="London start hour",
    yaxis_title="Shocks (all worlds)",
)
apply_notebook_style(hour_figure, height=360)

In [18]:
shocks_by_hour.T

start_hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
shock_class,,,,,,,,,,,,,,,,,,,,,
big,22,19,27,28,18,24,29,22,26,0,...,0,17,22,24,24,18,24,22,24,23
mild,64,62,77,56,71,66,58,65,46,0,...,0,69,63,63,55,65,66,69,62,72


**Known versus surprise, across every slot.** A known shock is already in the day-ahead price,
so the intraday close has nothing new to add on average. An upward surprise pushes the close
above day-ahead; a downward one pulls it below.

In [19]:
known_gw, surprise_gw = market.known_shock_gw, market.surprise_shock_gw
slot_kinds = {
    "no shock": (known_gw == 0) & (surprise_gw == 0),
    "known shock only": (known_gw != 0) & (surprise_gw == 0),
    "surprise up": surprise_gw > 0,
    "surprise down": surprise_gw < 0,
}
shock_kind_table = pd.DataFrame(
    [
        {
            "slot kind": name,
            "share of slots": mask.mean(),
            "mean close - day-ahead £/MWh": (close - day_ahead)[mask].mean(),
            "mean imbalance - day-ahead £/MWh": (imbalance - day_ahead)[mask].mean(),
        }
        for name, mask in slot_kinds.items()
    ]
).round(3)
shock_kind_table

,slot kind,share of slots,mean close - day-ahead £/MWh,mean imbalance - day-ahead £/MWh
0,no shock,0.833,-0.355,0.574
1,known shock only,0.069,-0.396,0.371
2,surprise up,0.049,19.242,31.993
3,surprise down,0.049,-12.596,-22.743


In [20]:
def biggest_study_shock(known: bool) -> pd.Series:
    """The largest upward big shock that starts in the study, known day-ahead or a surprise."""
    candidates = shocks.query(
        "shock_class == 'big' and direction == 'up' and known_day_ahead == @known"
        " and start_slot_index >= @WARMUP_DAYS * 48 and start_slot_index < @SLOT_COUNT - 16"
    )
    return candidates.loc[candidates["size_gw"].idxmax()]


example_shocks = {
    "surprise": biggest_study_shock(False),
    "known day-ahead": biggest_study_shock(True),
}
example_figure = make_subplots(
    rows=1, cols=2, subplot_titles=[f"Big upward shock, {name}" for name in example_shocks]
)
for column, (name, shock) in enumerate(example_shocks.items(), start=1):
    around = slice(
        max(shock["start_slot_index"] - 8, 0),
        min(shock["start_slot_index"] + shock["duration_slots"] + 8, SLOT_COUNT),
    )
    for label, values in (
        ("day-ahead", day_ahead),
        ("intraday close", close),
        ("imbalance", imbalance),
    ):
        example_figure.add_trace(
            go.Scatter(
                x=london[around],
                y=values[shock["world_id"], around].round(2),
                mode="lines+markers",
                name=label,
                legendgroup=label,
                showlegend=column == 1,
                marker={"size": 4},
            ),
            row=1,
            col=column,
        )
example_figure.update_layout(title="Two big upward shocks: what each price saw")
example_figure.update_yaxes(title_text="£/MWh", col=1)
apply_notebook_style(example_figure, height=420)

In [21]:
pd.DataFrame(example_shocks).T[
    ["world_id", "start_utc", "duration_slots", "size_gw", "direction", "known_day_ahead"]
]

,world_id,start_utc,duration_slots,size_gw,direction,known_day_ahead
surprise,36,2026-01-17 23:00:00+00:00,3,15.0,up,False
known day-ahead,9,2026-01-15 07:00:00+00:00,4,12.78824,up,True


## 5. Day-ahead, then intraday, then imbalance

- **Day-ahead** prices for every half-hour of London day D are published at 13:00 on D-1
  (section 8).
- **Intraday**: from publication until gate closure (`TRADING["gate_closure_minutes"]`, an hour
  before delivery by default), the price moves at each whole hour by a zero-mean update; updates
  for neighbouring half-hours are correlated, because news about an evening moves all of it.
  Surprises join from their reveal. The price at gate closure is the **intraday close**.
- **Imbalance**: the close, plus any surprise revealed only after gate closure, plus a premium set
  by whether the system is long or short (a two-state Markov chain; a surprise shifts the odds in
  its own direction), plus a fat Student-t tail.
- Every price is clipped to the auction floor and cap.

The chart follows one evening half-hour in several simulated weeks, from publication to the close.
The x-axis counts whole hours before gate closure.

In [22]:
SLOT_DATE = START_LOCAL_DATE.toordinal() + 1  # the Tuesday of the study
slot = int(
    np.flatnonzero(
        (np.asarray([d.toordinal() for d in london_date]) == SLOT_DATE)
        & (london.hour == 18)
        & (london.minute == 0)
    )[0]
)
published_at = pd.Timestamp(day_ahead_publication_utc_ns(run_starts[[slot]])[0], tz="UTC")
gate_closure = run_starts[slot] - pd.Timedelta(minutes=price_assumptions["gate_closure_minutes"])
hours_open = int((gate_closure - published_at) / pd.Timedelta(hours=1))
path = market.intraday_path_gbp_per_mwh
steps = np.arange(path.shape[2])
path_figure = go.Figure()
for world_id in range(8):
    path_figure.add_trace(
        go.Scatter(
            x=steps[: hours_open + 1],
            y=path[world_id, slot, : hours_open + 1].round(2),
            mode="lines",
            name=f"week {world_id}",
        )
    )
path_figure.update_xaxes(autorange="reversed")
path_figure.update_layout(
    title=(
        f"Intraday price of {london[slot]:%a %d %b %H:%M} London, from publication "
        f"({hours_open} h before gate closure) to the close"
    ),
    xaxis_title="Whole hours before gate closure (0 = the close)",
    yaxis_title="£/MWh",
)
apply_notebook_style(path_figure, height=420)

In [23]:
path_table = pd.DataFrame(
    {
        "week": range(8),
        "day-ahead": day_ahead[:8, slot],
        "intraday at publication": path[:8, slot, hours_open],
        "intraday close": close[:8, slot],
        "imbalance": imbalance[:8, slot],
        "system long": system_long[:8, slot],
    }
).round(2)
path_table

,week,day-ahead,intraday at publication,intraday close,imbalance,system long
0,0,151.96,151.96,141.49,127.97,True
1,1,102.47,102.47,85.58,64.81,True
2,2,103.67,103.67,88.58,64.42,True
3,3,58.42,58.42,44.14,31.68,True
4,4,78.25,78.25,79.27,90.85,False
5,5,167.54,167.54,178.04,187.42,False
6,6,120.98,120.98,106.84,100.39,True
7,7,127.23,127.23,138.03,107.23,True


**Imbalance, by system state.** The measured long share and premiums sit near their records
(which were calibrated to Elexon's system prices); surprises nudge the share. The tail
quantiles show the fat Student-t tail around the premium.

In [24]:
premium_by_state = np.where(
    system_long,
    price_assumptions["imbalance_long_premium_gbp_per_mwh"],
    price_assumptions["imbalance_short_premium_gbp_per_mwh"],
)
imbalance_table = pd.DataFrame(
    [
        {
            "state": state,
            "simulated share": mask.mean(),
            "record share": share,
            "simulated mean imbalance - close £/MWh": (imbalance - close)[mask].mean(),
            "record premium £/MWh": premium,
        }
        for state, mask, share, premium in (
            (
                "long",
                system_long,
                price_assumptions["imbalance_long_share"],
                price_assumptions["imbalance_long_premium_gbp_per_mwh"],
            ),
            (
                "short",
                ~system_long,
                1 - price_assumptions["imbalance_long_share"],
                price_assumptions["imbalance_short_premium_gbp_per_mwh"],
            ),
        )
    ]
).round(3)
tail = (imbalance - close - premium_by_state).ravel()
tail_quantiles = pd.Series(
    np.quantile(tail, [0.01, 0.05, 0.5, 0.95, 0.99]), index=["P1", "P5", "P50", "P95", "P99"]
).round(2)
display(imbalance_table)
tail_quantiles.rename("imbalance - close - state premium, £/MWh").to_frame().T

,state,simulated share,record share,simulated mean imbalance - close £/MWh,record premium £/MWh
0,long,0.52,0.53,-16.740,-16.8
1,short,0.48,0.47,20.146,20.2


,P1,P5,P50,P95,P99
"imbalance - close - state premium, £/MWh",-36.31,-18.7,-0.02,18.84,35.9


In [25]:
floor, cap = (
    price_assumptions["price_floor_gbp_per_mwh"],
    price_assumptions["price_cap_gbp_per_mwh"],
)
price_series = {
    "day-ahead": day_ahead,
    "intraday (every step)": path,
    "intraday close": close,
    "imbalance": imbalance,
}
cap_floor_table = pd.DataFrame(
    [
        {
            "price": name,
            "min £/MWh": values.min(),
            "max £/MWh": values.max(),
            "negative share": (values < 0).mean(),
            "at floor": int((values == floor).sum()),
            "at cap": int((values == cap).sum()),
            "floor (record)": floor,
            "cap (record)": cap,
        }
        for name, values in price_series.items()
    ]
).round(4)
cap_floor_table

,price,min £/MWh,max £/MWh,negative share,at floor,at cap,floor (record),cap (record)
0,day-ahead,-53.4137,334.0407,0.0174,0,0,-500.0,4000.0
1,intraday (every step),-63.0398,928.8714,0.0187,0,0,-500.0,4000.0
2,intraday close,-60.7537,926.9808,0.0225,0,0,-500.0,4000.0
3,imbalance,-155.5008,954.2092,0.0370,0,0,-500.0,4000.0


## 6. The no-free-money check

If the day-ahead price were the plain curve price, the convex curve would make zero-mean surprises
*raise* the average close (a convex function of a spread-out quantity has a higher mean than the
function of its mean), and a trader could earn by always buying day-ahead. So the day-ahead price
is built as the **expected intraday close given day-ahead information**, less a deliberate premium
(`da_id_premium_gbp_per_mwh`, default 0, decision 0004 item 58). The expectation is exact over the
surprise distribution (enumerated on a 0.1 GW grid, no random draw).

The check: across the study, mean(close - day-ahead) should equal the premium within sampling
error. The standard error is taken across simulated weeks (one mean per week first), because
half-hours inside a week are correlated. Try `PRICE_OVERRIDES = {"da_id_premium_gbp_per_mwh": 5.0}`.

In [26]:
per_week_gap = (close - day_ahead)[:, STUDY].mean(axis=1)
gap_mean = per_week_gap.mean()
gap_se = per_week_gap.std(ddof=1) / np.sqrt(WORLD_COUNT)
premium = price_assumptions["da_id_premium_gbp_per_mwh"]
no_free_money = pd.DataFrame(
    [
        {
            "premium (record) £/MWh": premium,
            "mean close - day-ahead £/MWh": gap_mean,
            "standard error £/MWh": gap_se,
            "difference in SEs": (gap_mean - premium) / gap_se,
            "weeks": WORLD_COUNT,
        }
    ]
).round(3)
no_free_money

,premium (record) £/MWh,mean close - day-ahead £/MWh,standard error £/MWh,difference in SEs,weeks
0,0.0,0.189,0.435,0.435,40


## 7. Comparison with the Elexon calibration (reference only)

`docs/research/elexon-price-calibration.md` fitted a year of Elexon half-hourly market index and
system prices (Contains BMRS data © Elexon Limited copyright and database right 2026). The
notebook reads its result tables straight from that file (no data is fetched) and sets the same
statistics from these synthetic weeks beside them.

This is a reference, not a test. Elexon's statistics cover twelve months and every season; these
weeks share one start date and one season, and each has only about two weeks of days, so a
day-to-day persistence estimate from them comes out low. The records say which values were
calibrated to this fit and which were tuned by hand (the comment above `_MARKET_SOURCE` in
`assumptions.py`).

In [27]:
repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
elexon_text = (repo_root / "docs/research/elexon-price-calibration.md").read_text()


def markdown_table(heading: str) -> pd.DataFrame:
    """The first pipe table after a heading line of the Elexon calibration note, as strings."""
    after = elexon_text.split(heading, 1)[1].splitlines()
    rows = []
    for line in after:
        if line.startswith("|"):
            rows.append([cell.strip() for cell in line.strip("|").split("|")])
        elif rows:
            break
    return pd.DataFrame(rows[2:], columns=rows[0])


def number(text: str) -> float:
    """The first number in a table cell such as '£29.95/MWh', '+£20.18/MWh' or '2.12%'."""
    return float(re.search(r"[-+]?\d+(?:\.\d+)?", text.replace("£", "")).group())


elexon = pd.concat(
    [
        markdown_table("### Daily-mean level"),
        markdown_table("### Half-hour residual noise").assign(
            stat=lambda frame: "residual " + frame["stat"]
        ),
        markdown_table("### System (imbalance) price vs market index price"),
    ],
    ignore_index=True,
)
elexon["Elexon (observed)"] = elexon["value"].map(number)

# The same statistics from the synthetic day-ahead prices: complete London days only.
frame = pd.DataFrame(
    {
        "world": np.repeat(np.arange(WORLD_COUNT), SLOT_COUNT),
        "date": np.tile(london_date, WORLD_COUNT),
        "half_hour": np.tile(london.hour * 2 + london.minute // 30, WORLD_COUNT),
        "weekend": np.tile(london.dayofweek >= 5, WORLD_COUNT),
        "price": day_ahead.ravel(),
    }
)
slots_per_day = frame.groupby(["world", "date"])["price"].transform("size")
frame = frame[slots_per_day == 48].copy()
daily_mean = frame.groupby(["world", "date"])["price"].mean()
frame["residual"] = frame["price"] - frame.groupby(["world", "date"])["price"].transform("mean")
frame["residual"] -= frame.groupby(["weekend", "half_hour"])["residual"].transform("mean")
daily_matrix = daily_mean.unstack("date").to_numpy()
residual_matrix = np.stack([group.to_numpy() for _, group in frame.groupby("world")["residual"]])

model_value = {
    "SD of daily means": daily_mean.std(),
    "AR(1) coefficient (lag-1, on daily means)": lag_one(daily_matrix),
    "Negative-price half-hour share": 100 * (day_ahead < 0).mean(),
    "Daily-mean P10": daily_mean.quantile(0.10),
    "Daily-mean P50": daily_mean.quantile(0.50),
    "Daily-mean P90": daily_mean.quantile(0.90),
    "Daily-mean P90 - P10": daily_mean.quantile(0.90) - daily_mean.quantile(0.10),
    "residual AR(1) coefficient (lag-1)": lag_one(residual_matrix),
    "residual SD": frame["residual"].std(),
    # Elexon's NIV > 0 is "short" (see the note's sign convention); the model's premium is
    # imbalance minus the intraday close, the nearest match to its SIP minus MIP.
    "Share of periods with NIV > 0": 100 * (~system_long).mean(),
    "Share of periods with NIV < 0": 100 * system_long.mean(),
    "Mean SIP - MIP premium when NIV > 0": (imbalance - close)[~system_long].mean(),
    "Mean SIP - MIP premium when NIV < 0": (imbalance - close)[system_long].mean(),
}
elexon["this model (synthetic)"] = elexon["stat"].map(model_value)
elexon[["stat", "Elexon (observed)", "this model (synthetic)"]].round(3)

,stat,Elexon (observed),this model (synthetic)
0,SD of daily means,29.950,32.637
1,"AR(1) coefficient (lag-1, on daily means)",0.644,0.310
2,Negative-price half-hour share,2.120,1.745
3,Daily-mean P10,57.530,39.902
4,Daily-mean P50,93.110,84.092
5,Daily-mean P90,134.730,124.110
6,Daily-mean P90 - P10,77.190,84.207
7,residual AR(1) coefficient (lag-1),0.920,0.851
8,residual SD,21.610,19.721
9,Share of periods with NIV > 0,47.200,47.972


In [28]:
comparison_figure = go.Figure()
shown = elexon[~elexon["stat"].str.contains("Share|AR|share")]
for column in ("Elexon (observed)", "this model (synthetic)"):
    comparison_figure.add_trace(go.Bar(x=shown["stat"], y=shown[column].round(2), name=column))
comparison_figure.update_layout(
    barmode="group",
    title="Price statistics in £/MWh: Elexon year against these synthetic weeks",
    yaxis_title="£/MWh",
)
apply_notebook_style(comparison_figure, height=440)

The table above is the paired text version of this chart (the shares and AR(1) coefficients,
which are not in £/MWh, appear only in the table).

## 8. Publication at 13:00 the day before, and what a plan may see

Every half-hour of London delivery day D is published at 13:00 London on D-1
(`PRICES["day_ahead_publication_local_hour"]`, plan B4, decision 0004 item 53); the generator
stamps it on every day-ahead row as `forecast_available_at_utc`.

`action.visible_day_ahead_prices` is the single rule for what a decision may rank: a published
slot keeps its price; an unpublished one gets the **expected shape**, the mean published price
of the same London half-hour over the most recent published days. An evening plug-in sees every
price to the next morning; a plan made before 13:00 ranks tomorrow on the shape. The smart
charger and the trading overlay both call it, so they see the same prices.

In [29]:
publication = (
    market.day_ahead.query("world_id == 0")
    .assign(
        delivery_date=lambda f: f["interval_start_utc"].dt.tz_convert(LONDON).dt.date,
        published_london=lambda f: f["forecast_available_at_utc"].dt.tz_convert(LONDON),
    )
    .groupby("delivery_date", as_index=False)["published_london"]
    .agg(["min", "max"])
)
publication

,delivery_date,min,max
0,2026-01-05,2026-01-04 13:00:00+00:00,2026-01-04 13:00:00+00:00
1,2026-01-06,2026-01-05 13:00:00+00:00,2026-01-05 13:00:00+00:00
2,2026-01-07,2026-01-06 13:00:00+00:00,2026-01-06 13:00:00+00:00
3,2026-01-08,2026-01-07 13:00:00+00:00,2026-01-07 13:00:00+00:00
4,2026-01-09,2026-01-08 13:00:00+00:00,2026-01-08 13:00:00+00:00
5,2026-01-10,2026-01-09 13:00:00+00:00,2026-01-09 13:00:00+00:00
6,2026-01-11,2026-01-10 13:00:00+00:00,2026-01-10 13:00:00+00:00
7,2026-01-12,2026-01-11 13:00:00+00:00,2026-01-11 13:00:00+00:00
8,2026-01-13,2026-01-12 13:00:00+00:00,2026-01-12 13:00:00+00:00
9,2026-01-14,2026-01-13 13:00:00+00:00,2026-01-13 13:00:00+00:00


In [30]:
tuesday = date.fromordinal(START_LOCAL_DATE.toordinal() + 1)
wednesday = date.fromordinal(START_LOCAL_DATE.toordinal() + 2)
before_publication = pd.Timestamp(datetime.combine(tuesday, time(10)), tz=LONDON)
after_publication = pd.Timestamp(datetime.combine(tuesday, time(14)), tz=LONDON)
visible_before = visible_day_ahead_prices(day_ahead, run_starts, before_publication)
visible_after = visible_day_ahead_prices(day_ahead, run_starts, after_publication)
wednesday_slots = london_date == wednesday
tuesday_slots = london_date == tuesday

visible_figure = go.Figure()
for label, values, dash in (
    ("day-ahead as published", day_ahead, "solid"),
    ("visible at Tue 14:00 (published)", visible_after, "dot"),
    ("visible at Tue 10:00 (expected shape)", visible_before, "dash"),
):
    visible_figure.add_trace(
        go.Scatter(
            x=london[wednesday_slots],
            y=values[world, wednesday_slots].round(2),
            mode="lines",
            line={"dash": dash},
            name=label,
        )
    )
visible_figure.update_layout(
    title=f"Wednesday's prices as seen from Tuesday morning and afternoon, week {world}",
    xaxis_title="London local time",
    yaxis_title="£/MWh",
)
apply_notebook_style(visible_figure, height=400)

In [31]:
pd.DataFrame(
    [
        {
            "series": label,
            "mean £/MWh": values[world, wednesday_slots].mean(),
            "min £/MWh": values[world, wednesday_slots].min(),
            "cheapest half-hour (London)": london[wednesday_slots][
                values[world, wednesday_slots].argmin()
            ].strftime("%H:%M"),
        }
        for label, values in (
            ("day-ahead as published", day_ahead),
            ("visible at Tue 14:00", visible_after),
            ("visible at Tue 10:00", visible_before),
        )
    ]
).round(2)

,series,mean £/MWh,min £/MWh,cheapest half-hour (London)
0,day-ahead as published,90.42,47.73,12:00
1,visible at Tue 14:00,90.42,47.73,12:00
2,visible at Tue 10:00,113.38,79.68,03:00


## 9. Sanity checks

Each check restates one property the text above relies on. If a model change breaks one, the
notebook test fails here rather than leaving the prose wrong.

In [32]:
perturbed = day_ahead.copy()
perturbed[:, wednesday_slots] += 1000.0
hour_minutes = shock_start_hour.dt.hour + shock_start_hour.dt.minute / 60
window_start = price_assumptions["shock_window_start_local_hour"]
window_end = price_assumptions["shock_window_end_local_hour"]
heating_threshold = price_assumptions["heating_threshold_c"]
checks = {
    "net demand = demand + heating - wind - solar + known shocks": decomposition_gap_gw < 1e-9,
    "heating is zero above its threshold": bool(
        (hinge.loc[hinge["temperature_c"] > heating_threshold, "heating_gw"] == 0).all()
    ),
    "supply curve is convex: a bigger jump from a tighter start": bool(
        np.all(np.diff(increments["price change £/MWh"]) > 0)
    ),
    "daily level is one value per London day": level_spread_within_day < 0.05,
    "half-hour noise SD within 10% of its record": abs(
        noise_gbp.std() / price_assumptions["day_ahead_sd_gbp_per_mwh"] - 1
    )
    < 0.10,
    "half-hour noise lag-one within 0.05 of its record": abs(
        lag_one(noise_gbp) - assumptions.PRICES["ar1_phi"].value
    )
    < 0.05,
    "every shock starts inside the nightly window": bool(
        ((hour_minutes >= window_start) | (hour_minutes < window_end)).all()
    ),
    "upward surprises lift the close above day-ahead": shock_kind_table.set_index("slot kind").at[
        "surprise up", "mean close - day-ahead £/MWh"
    ]
    > 0,
    "downward surprises pull the close below day-ahead": shock_kind_table.set_index("slot kind").at[
        "surprise down", "mean close - day-ahead £/MWh"
    ]
    < 0,
    "intraday path starts at day-ahead plus the premium": np.allclose(
        path[:, slot, hours_open + 1 :], (day_ahead[:, slot] + premium)[:, None]
    ),
    "intraday path ends at the close": np.array_equal(path[:, :, 0], close),
    "no free money: mean close - day-ahead is the premium within 4 SE": abs(gap_mean - premium)
    < 4 * gap_se,
    "every price within the floor and cap": all(
        values.min() >= floor and values.max() <= cap for values in price_series.values()
    ),
    "day-ahead published 13:00 London the day before": bool(
        (
            (publication["min"] == publication["max"])
            & (publication["min"].dt.hour == price_assumptions["day_ahead_publication_local_hour"])
            & (
                publication["min"].dt.date.map(date.toordinal) + 1
                == publication["delivery_date"].map(date.toordinal)
            )
        ).all()
    ),
    "a published day is seen as published": np.array_equal(
        visible_before[:, tuesday_slots], day_ahead[:, tuesday_slots]
    )
    and np.array_equal(visible_after[:, wednesday_slots], day_ahead[:, wednesday_slots]),
    "an unpublished day is ranked on the shape, not its price": not np.array_equal(
        visible_before[:, wednesday_slots], day_ahead[:, wednesday_slots]
    ),
    "changing an unpublished price changes nothing visible": np.array_equal(
        visible_day_ahead_prices(perturbed, run_starts, before_publication), visible_before
    ),
    "every price is labelled synthetic": set(market.day_ahead["evidence_kind"])
    == set(market.realised["evidence_kind"])
    == {"synthetic"},
    "the Elexon tables were read": elexon["Elexon (observed)"].notna().all() and len(elexon) == 13,
}
check_table = pd.Series(checks, name="passes").to_frame()
assert check_table["passes"].all(), check_table[~check_table["passes"]]
check_table

,passes
net demand = demand + heating - wind - solar + known shocks,True
heating is zero above its threshold,True
supply curve is convex: a bigger jump from a tighter start,True
daily level is one value per London day,True
half-hour noise SD within 10% of its record,True
half-hour noise lag-one within 0.05 of its record,True
every shock starts inside the nightly window,True
upward surprises lift the close above day-ahead,True
downward surprises pull the close below day-ahead,True
intraday path starts at day-ahead plus the premium,True


## 10. Limitations

- Every price is synthetic. The demand shapes are shapes that reproduce Elexon *price* patterns
  through this supply curve, not metered demand; the supply curve, heating, wind, solar and shock
  values are illustrative choices.
- One season per run, chosen by the start month; no gas-price, carbon or interconnector model
  beyond the daily level.
- Wholesale prices are national: no locational prices or network constraints in the price.
- The NIV state is independent of prices except for the surprise shift.
- The no-free-money expectation ignores the cap acting on the zero-mean intraday updates; at the
  default cap this never binds in these weeks.
- Elexon's figures are from one year (September 2025 to September 2026) and are a reference, not
  a fit these weeks are tested against.